# Smart Grid Digital Twin & Power Distribution Simulator

**Version:** 1.0.0  
**Language:** Python  
**Environment:** Google Colab  
**Simulation Type:** Fully Simulated

## Project Overview

The Smart Grid Digital Twin is a Python-based simulation platform designed to model electrical power generation, transmission, distribution, energy storage, and grid monitoring.

The system simulates interactions between power generation facilities, substations, transformers, and electricity consumers. It incorporates automated grid diagnostics, fault management, historical telemetry storage, machine-learning-based anomaly detection, and interactive engineering visualizations.

## Core Capabilities

- Electrical grid topology modeling using NetworkX
- Power generation and demand simulation
- Battery energy storage and generation balancing
- Transformer loading and diagnostics
- Automated fault prioritization and response workflows
- Historical telemetry storage using SQLite
- Machine learning anomaly detection using Scikit-learn
- Interactive engineering dashboards using Plotly
- Three-dimensional infrastructure visualization

## Technology Stack

Python | NumPy | Pandas | NetworkX | Scikit-learn | SQLite | Plotly

## Engineering Disclaimer

This project is an educational simulation. Electrical behavior, power-flow calculations, protection systems, and grid operating conditions are simplified and are not intended for real-world utility operations.

In [ ]:
# Smart Grid Digital Twin
# Phase 1: Environment Setup

!pip -q install numpy pandas plotly networkx scikit-learn dash jupyter-dash

print("Smart Grid Digital Twin environment initialized.")
print("Core simulation and visualization libraries are ready.")


import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px

from sklearn.ensemble import IsolationForest
from datetime import datetime, timedelta
import sqlite3
import json
import random
import math
import warnings

warnings.filterwarnings("ignore")

np.random.seed(42)
random.seed(42)

PROJECT_NAME = "Smart Grid Digital Twin"
PROJECT_VERSION = "1.0.0"
SIMULATION_MODE = "FULLY_SIMULATED"

SIMULATION_CONFIG = {
    "nominal_frequency_hz": 60.0,
    "nominal_voltage_kv": 230.0,
    "simulation_interval_seconds": 1,
    "initial_grid_load_mw": 850.0,
    "maximum_grid_capacity_mw": 1200.0,
    "voltage_tolerance_percent": 5.0,
    "frequency_tolerance_hz": 0.5
}

print(f"Project: {PROJECT_NAME}")
print(f"Version: {PROJECT_VERSION}")
print(f"Simulation Mode: {SIMULATION_MODE}")
print(f"Nominal Frequency: {SIMULATION_CONFIG['nominal_frequency_hz']} Hz")
print(f"Nominal Voltage: {SIMULATION_CONFIG['nominal_voltage_kv']} kV")
print("Environment initialized successfully.")


# Define the electrical grid infrastructure

grid_components = {
    "generators": [
        {"id": "GEN-001", "name": "North Power Station", "type": "thermal", "capacity_mw": 450, "output_mw": 350},
        {"id": "GEN-002", "name": "South Solar Farm", "type": "solar", "capacity_mw": 250, "output_mw": 180},
        {"id": "GEN-003", "name": "West Wind Farm", "type": "wind", "capacity_mw": 300, "output_mw": 220}
    ],
    "substations": [
        {"id": "SUB-001", "name": "North Regional Substation", "voltage_kv": 230},
        {"id": "SUB-002", "name": "Central Substation", "voltage_kv": 230},
        {"id": "SUB-003", "name": "South Regional Substation", "voltage_kv": 115},
        {"id": "SUB-004", "name": "West Distribution Substation", "voltage_kv": 115}
    ],
    "transformers": [
        {"id": "TRF-001", "name": "North Transformer 1", "capacity_mva": 200},
        {"id": "TRF-002", "name": "North Transformer 2", "capacity_mva": 150},
        {"id": "TRF-003", "name": "Central Transformer 1", "capacity_mva": 200},
        {"id": "TRF-004", "name": "Central Transformer 2", "capacity_mva": 150},
        {"id": "TRF-005", "name": "South Transformer 1", "capacity_mva": 125},
        {"id": "TRF-006", "name": "West Transformer 1", "capacity_mva": 125}
    ],
    "consumers": [
        {"id": "LOAD-001", "name": "North Residential District", "demand_mw": 75},
        {"id": "LOAD-002", "name": "Central Business District", "demand_mw": 120},
        {"id": "LOAD-003", "name": "Industrial Manufacturing Plant", "demand_mw": 180},
        {"id": "LOAD-004", "name": "South Residential District", "demand_mw": 65},
        {"id": "LOAD-005", "name": "West Commercial District", "demand_mw": 90},
        {"id": "LOAD-006", "name": "Regional Hospital", "demand_mw": 35},
        {"id": "LOAD-007", "name": "Data Center", "demand_mw": 110},
        {"id": "LOAD-008", "name": "Water Treatment Facility", "demand_mw": 40}
    ]
}

component_counts = {
    component_type: len(components)
    for component_type, components in grid_components.items()
}

print("Grid infrastructure defined successfully.")
print(f"Generators: {component_counts['generators']}")
print(f"Substations: {component_counts['substations']}")
print(f"Transformers: {component_counts['transformers']}")
print(f"Consumers: {component_counts['consumers']}")
print(f"Total components: {sum(component_counts.values())}")



# Build the electrical grid network

grid = nx.Graph()

# Register infrastructure components
for category, components in grid_components.items():
    for component in components:
        grid.add_node(
            component["id"],
            category=category,
            **component
        )

# Define transmission and distribution connections
grid_connections = [
    ("GEN-001", "SUB-001", 230, 450, 0.015),
    ("GEN-002", "SUB-003", 115, 250, 0.020),
    ("GEN-003", "SUB-004", 115, 300, 0.018),
    ("SUB-001", "SUB-002", 230, 400, 0.012),
    ("SUB-002", "SUB-003", 115, 250, 0.025),
    ("SUB-002", "SUB-004", 115, 250, 0.022),
    ("SUB-001", "TRF-001", 230, 200, 0.010),
    ("SUB-001", "TRF-002", 230, 150, 0.012),
    ("SUB-002", "TRF-003", 230, 200, 0.010),
    ("SUB-002", "TRF-004", 230, 150, 0.012),
    ("SUB-003", "TRF-005", 115, 125, 0.015),
    ("SUB-004", "TRF-006", 115, 125, 0.015),
    ("TRF-001", "LOAD-001", 33, 100, 0.020),
    ("TRF-002", "LOAD-006", 33, 80, 0.025),
    ("TRF-003", "LOAD-002", 33, 150, 0.020),
    ("TRF-003", "LOAD-007", 33, 130, 0.018),
    ("TRF-004", "LOAD-003", 33, 200, 0.015),
    ("TRF-005", "LOAD-004", 33, 100, 0.020),
    ("TRF-006", "LOAD-005", 33, 120, 0.020),
    ("TRF-006", "LOAD-008", 33, 80, 0.025)
]

for source, destination, voltage, capacity, resistance in grid_connections:
    grid.add_edge(
        source,
        destination,
        voltage_kv=voltage,
        capacity_mw=capacity,
        resistance_pu=resistance,
        status="ONLINE"
    )

print("Electrical network constructed successfully.")
print(f"Network nodes: {grid.number_of_nodes()}")
print(f"Network connections: {grid.number_of_edges()}")
print(f"Network connected: {nx.is_connected(grid)}")
print(f"Network density: {nx.density(grid):.3f}")


# Visualize the electrical grid topology

node_positions = nx.spring_layout(grid, seed=42, k=1.8, iterations=100)

edge_x = []
edge_y = []

for source, destination in grid.edges():
    x0, y0 = node_positions[source]
    x1, y1 = node_positions[destination]

    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode="lines",
    line=dict(width=1.5, color="#64748b"),
    hoverinfo="none"
)

category_colors = {
    "generators": "#f59e0b",
    "substations": "#3b82f6",
    "transformers": "#8b5cf6",
    "consumers": "#10b981"
}

node_x = []
node_y = []
node_colors = []
node_labels = []
node_details = []

for node_id, attributes in grid.nodes(data=True):
    x, y = node_positions[node_id]

    node_x.append(x)
    node_y.append(y)
    node_colors.append(category_colors[attributes["category"]])
    node_labels.append(node_id)
    node_details.append(
        f"{attributes['name']}<br>"
        f"ID: {node_id}<br>"
        f"Category: {attributes['category']}"
    )

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers+text",
    text=node_labels,
    textposition="top center",
    hovertext=node_details,
    hoverinfo="text",
    marker=dict(
        size=18,
        color=node_colors,
        line=dict(width=1.5, color="white")
    )
)

fig = go.Figure(data=[edge_trace, node_trace])

fig.update_layout(
    title="Smart Grid Network Topology",
    template="plotly_dark",
    showlegend=False,
    hovermode="closest",
    height=750,
    margin=dict(l=20, r=20, t=60, b=20),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False)
)

fig.show()


# Initialize power generation models

generator_data = pd.DataFrame(grid_components["generators"])

generator_data["utilization_percent"] = (
    generator_data["output_mw"] /
    generator_data["capacity_mw"] * 100
)

generator_data["status"] = "ONLINE"

generator_data["available_capacity_mw"] = (
    generator_data["capacity_mw"] -
    generator_data["output_mw"]
)

print("Power generation models initialized.")
display(
    generator_data[
        ["id", "name", "type", "capacity_mw",
         "output_mw", "utilization_percent",
         "available_capacity_mw", "status"]
    ].round(2)
)

print(f"Total generation capacity: {generator_data['capacity_mw'].sum()} MW")
print(f"Current power generation: {generator_data['output_mw'].sum()} MW")


# Initialize electricity demand models

consumer_data = pd.DataFrame(grid_components["consumers"])

consumer_categories = {
    "LOAD-001": "Residential",
    "LOAD-002": "Commercial",
    "LOAD-003": "Industrial",
    "LOAD-004": "Residential",
    "LOAD-005": "Commercial",
    "LOAD-006": "Critical Infrastructure",
    "LOAD-007": "Critical Infrastructure",
    "LOAD-008": "Critical Infrastructure"
}

consumer_priorities = {
    "LOAD-001": "Normal",
    "LOAD-002": "Normal",
    "LOAD-003": "High",
    "LOAD-004": "Normal",
    "LOAD-005": "Normal",
    "LOAD-006": "Critical",
    "LOAD-007": "Critical",
    "LOAD-008": "High"
}

consumer_data["category"] = consumer_data["id"].map(consumer_categories)
consumer_data["priority"] = consumer_data["id"].map(consumer_priorities)
consumer_data["actual_consumption_mw"] = consumer_data["demand_mw"]
consumer_data["status"] = "ONLINE"

total_demand = consumer_data["demand_mw"].sum()

print("Electricity demand models initialized.")
display(consumer_data)

print(f"Total electricity demand: {total_demand} MW")
print(f"Critical infrastructure demand: {consumer_data.loc[consumer_data['priority'] == 'Critical', 'demand_mw'].sum()} MW")
print(f"Generation surplus: {generator_data['output_mw'].sum() - total_demand} MW")


# Calculate grid generation and demand balance

total_generation = generator_data["output_mw"].sum()
total_capacity = generator_data["capacity_mw"].sum()
total_demand = consumer_data["actual_consumption_mw"].sum()

power_balance = total_generation - total_demand

reserve_margin = (
    (total_capacity - total_demand) / total_demand
) * 100

generation_utilization = (
    total_generation / total_capacity
) * 100

grid_status = "STABLE" if power_balance >= 0 else "DEFICIT"

grid_metrics = {
    "total_generation_mw": total_generation,
    "total_capacity_mw": total_capacity,
    "total_demand_mw": total_demand,
    "power_balance_mw": power_balance,
    "reserve_margin_percent": reserve_margin,
    "generation_utilization_percent": generation_utilization,
    "grid_status": grid_status
}

print("Grid operational assessment")
print(f"Generation: {total_generation:.2f} MW")
print(f"Demand: {total_demand:.2f} MW")
print(f"Power balance: {power_balance:.2f} MW")
print(f"Reserve margin: {reserve_margin:.2f}%")
print(f"Generation utilization: {generation_utilization:.2f}%")
print(f"Grid status: {grid_status}")


# Simulate changing electricity demand

def simulate_demand(hour, base_demand, category):
    hour = hour % 24

    if category == "Residential":
        multiplier = 0.85 + 0.20 * np.exp(-((hour - 19) / 3) ** 2)

    elif category == "Commercial":
        multiplier = 0.75 + 0.30 * np.exp(-((hour - 13) / 4) ** 2)

    elif category == "Industrial":
        multiplier = 1.0

    elif category == "Critical Infrastructure":
        multiplier = 0.98

    else:
        multiplier = 1.0

    variation = np.random.normal(1.0, 0.015)

    return max(0, base_demand * multiplier * variation)


simulation_hour = 18

consumer_data["actual_consumption_mw"] = consumer_data.apply(
    lambda row: simulate_demand(
        simulation_hour,
        row["demand_mw"],
        row["category"]
    ),
    axis=1
)

consumer_data["demand_change_mw"] = (
    consumer_data["actual_consumption_mw"] -
    consumer_data["demand_mw"]
)

print(f"Demand simulation completed for hour {simulation_hour}:00")
display(
    consumer_data[
        ["id", "name", "demand_mw",
         "actual_consumption_mw", "demand_change_mw"]
    ].round(2)
)

print(
    f"Updated total demand: "
    f"{consumer_data['actual_consumption_mw'].sum():.2f} MW"
)


# Visualize electricity demand by consumer

fig = go.Figure()

fig.add_trace(go.Bar(
    name="Baseline Demand",
    x=consumer_data["name"],
    y=consumer_data["demand_mw"],
    marker_color="#64748b"
))

fig.add_trace(go.Bar(
    name="Actual Consumption",
    x=consumer_data["name"],
    y=consumer_data["actual_consumption_mw"],
    marker_color="#06b6d4"
))

fig.update_layout(
    title="Electricity Demand by Consumer",
    xaxis_title="Grid Consumer",
    yaxis_title="Power Demand (MW)",
    barmode="group",
    template="plotly_dark",
    height=550,
    xaxis_tickangle=-35,
    legend_title="Measurement"
)

fig.show()


# Update grid operating metrics

total_generation = generator_data["output_mw"].sum()
total_capacity = generator_data["capacity_mw"].sum()
total_demand = consumer_data["actual_consumption_mw"].sum()

power_balance = total_generation - total_demand

reserve_margin = (
    (total_capacity - total_demand) / total_demand
) * 100

generation_utilization = (
    total_generation / total_capacity
) * 100

grid_status = "STABLE" if power_balance >= 0 else "DEFICIT"

grid_metrics.update({
    "total_generation_mw": total_generation,
    "total_capacity_mw": total_capacity,
    "total_demand_mw": total_demand,
    "power_balance_mw": power_balance,
    "reserve_margin_percent": reserve_margin,
    "generation_utilization_percent": generation_utilization,
    "grid_status": grid_status
})

print("Updated Grid Operating Metrics")
print(f"Total Generation: {total_generation:.2f} MW")
print(f"Total Demand: {total_demand:.2f} MW")
print(f"Available Power Surplus: {power_balance:.2f} MW")
print(f"Reserve Margin: {reserve_margin:.2f}%")
print(f"Generation Utilization: {generation_utilization:.2f}%")
print(f"Operating Status: {grid_status}")

def simulate_generation(hour):
    hour = hour % 24

    generation = {}

    for _, generator in generator_data.iterrows():
        generator_id = generator["id"]
        capacity = generator["capacity_mw"]

        if generator["type"] == "thermal":
            output = capacity * 0.78

        elif generator["type"] == "solar":
            solar_factor = max(0, np.sin(np.pi * (hour - 6) / 12))
            output = capacity * solar_factor * 0.85

        elif generator["type"] == "wind":
            wind_factor = np.random.uniform(0.55, 0.85)
            output = capacity * wind_factor

        else:
            output = 0

        generation[generator_id] = min(capacity, max(0, output))

    return generation


simulation_hour = 18

generation_output = simulate_generation(simulation_hour)

generator_data["output_mw"] = generator_data["id"].map(generation_output)

generator_data["utilization_percent"] = (
    generator_data["output_mw"] / generator_data["capacity_mw"]
) * 100

print(f"Generator Output at Simulation Hour {simulation_hour}:")
print(
    generator_data[
        ["id", "name", "type", "capacity_mw", "output_mw", "utilization_percent"]
    ].round(2).to_string(index=False)
)

print(f"\nTotal Generation: {generator_data['output_mw'].sum():.2f} MW")

total_generation = generator_data["output_mw"].sum()
total_capacity = generator_data["capacity_mw"].sum()
total_demand = consumer_data["actual_consumption_mw"].sum()

power_balance = total_generation - total_demand

reserve_margin = (
    (total_capacity - total_demand) / total_demand
) * 100

generation_utilization = (
    total_generation / total_capacity
) * 100

grid_status = "STABLE" if power_balance >= 0 else "DEFICIT"

grid_metrics.update({
    "total_generation_mw": total_generation,
    "total_capacity_mw": total_capacity,
    "total_demand_mw": total_demand,
    "power_balance_mw": power_balance,
    "reserve_margin_percent": reserve_margin,
    "generation_utilization_percent": generation_utilization,
    "grid_status": grid_status
})

print("Updated Grid Operating Conditions")
print(f"Total Generation: {total_generation:.2f} MW")
print(f"Total Demand: {total_demand:.2f} MW")
print(f"Power Balance: {power_balance:.2f} MW")
print(f"Reserve Margin: {reserve_margin:.2f}%")
print(f"Generation Utilization: {generation_utilization:.2f}%")
print(f"Operating Status: {grid_status}")

def balance_generation(demand_mw):
    thermal_generator = generator_data[
        generator_data["type"] == "thermal"
    ].iloc[0]

    thermal_capacity = thermal_generator["capacity_mw"]

    renewable_generation = generator_data[
        generator_data["type"].isin(["solar", "wind"])
    ]["output_mw"].sum()

    required_thermal_output = demand_mw - renewable_generation

    adjusted_thermal_output = min(
        thermal_capacity,
        max(0, required_thermal_output)
    )

    generator_data.loc[
        generator_data["type"] == "thermal",
        "output_mw"
    ] = adjusted_thermal_output

    generator_data["utilization_percent"] = (
        generator_data["output_mw"] /
        generator_data["capacity_mw"]
    ) * 100

    new_generation = generator_data["output_mw"].sum()
    remaining_shortage = max(0, demand_mw - new_generation)

    return new_generation, remaining_shortage


new_generation, remaining_shortage = balance_generation(total_demand)

print("Automatic Generation Control Results")
print(f"Demand: {total_demand:.2f} MW")
print(f"Adjusted Generation: {new_generation:.2f} MW")
print(f"Remaining Shortage: {remaining_shortage:.2f} MW")

print("\nUpdated Generator Dispatch:")
print(
    generator_data[
        ["id", "name", "output_mw", "capacity_mw", "utilization_percent"]
    ].round(2).to_string(index=False)
)

storage_system = {
    "id": "BESS-001",
    "name": "Central Grid Battery Storage",
    "capacity_mwh": 200.0,
    "state_of_charge_percent": 80.0,
    "maximum_discharge_mw": 100.0,
    "maximum_charge_mw": 75.0,
    "status": "ONLINE"
}

def discharge_battery(shortage_mw, duration_hours=1):
    available_energy = (
        storage_system["capacity_mwh"] *
        storage_system["state_of_charge_percent"] / 100
    )

    available_power = min(
        storage_system["maximum_discharge_mw"],
        available_energy / duration_hours
    )

    discharge_mw = min(shortage_mw, available_power)

    energy_used = discharge_mw * duration_hours

    storage_system["state_of_charge_percent"] -= (
        energy_used / storage_system["capacity_mwh"]
    ) * 100

    storage_system["state_of_charge_percent"] = max(
        0, storage_system["state_of_charge_percent"]
    )

    return discharge_mw


battery_output = discharge_battery(remaining_shortage)

final_generation = new_generation + battery_output
remaining_shortage = max(0, total_demand - final_generation)

print("Battery Energy Storage Dispatch")
print(f"Battery Output: {battery_output:.2f} MW")
print(f"Remaining Shortage: {remaining_shortage:.2f} MW")
print(
    f"Battery State of Charge: "
    f"{storage_system['state_of_charge_percent']:.2f}%"
)
print(f"Total Available Supply: {final_generation:.2f} MW")

total_generation = generator_data["output_mw"].sum()
total_supply = total_generation + battery_output
total_demand = consumer_data["actual_consumption_mw"].sum()

power_balance = total_supply - total_demand

if power_balance > 0:
    grid_status = "SURPLUS"
elif power_balance < 0:
    grid_status = "DEFICIT"
else:
    grid_status = "BALANCED"

grid_metrics.update({
    "total_generation_mw": total_generation,
    "total_supply_mw": total_supply,
    "total_demand_mw": total_demand,
    "power_balance_mw": power_balance,
    "grid_status": grid_status,
    "battery_output_mw": battery_output,
    "battery_charge_percent": storage_system["state_of_charge_percent"]
})

print("Smart Grid Operating Dashboard")
print(f"Generator Production: {total_generation:.2f} MW")
print(f"Battery Contribution: {battery_output:.2f} MW")
print(f"Total Available Supply: {total_supply:.2f} MW")
print(f"Electricity Demand: {total_demand:.2f} MW")
print(f"Power Balance: {power_balance:.2f} MW")
print(f"Battery Charge: {storage_system['state_of_charge_percent']:.2f}%")
print(f"Grid Operating Status: {grid_status}")

transformer_connections = {
    "TRF-001": ["LOAD-001"],
    "TRF-002": ["LOAD-006"],
    "TRF-003": ["LOAD-002", "LOAD-007"],
    "TRF-004": ["LOAD-003"],
    "TRF-005": ["LOAD-004"],
    "TRF-006": ["LOAD-005", "LOAD-008"]
}

transformer_data = pd.DataFrame(grid_components["transformers"])

transformer_loads = []

for _, transformer in transformer_data.iterrows():
    transformer_id = transformer["id"]
    connected_loads = transformer_connections[transformer_id]

    actual_load = consumer_data[
        consumer_data["id"].isin(connected_loads)
    ]["actual_consumption_mw"].sum()

    capacity = transformer["capacity_mva"]
    utilization = (actual_load / capacity) * 100

    if utilization > 100:
        status = "OVERLOADED"
    elif utilization >= 90:
        status = "WARNING"
    else:
        status = "NORMAL"

    transformer_loads.append({
        "transformer_id": transformer_id,
        "name": transformer["name"],
        "capacity_mva": capacity,
        "actual_load_mw": actual_load,
        "utilization_percent": utilization,
        "status": status
    })

transformer_data = pd.DataFrame(transformer_loads)

print("Transformer Loading Analysis")
print(
    transformer_data[
        ["transformer_id", "name", "capacity_mva",
         "actual_load_mw", "utilization_percent", "status"]
    ].round(2).to_string(index=False)
)

print(
    f"\nTransformers Requiring Attention: "
    f"{(transformer_data['status'] != 'NORMAL').sum()}"
)

def diagnose_transformer(row):
    if row["utilization_percent"] > 100:
        return "CRITICAL", "Reduce connected load or transfer demand to another transformer."
    elif row["utilization_percent"] >= 90:
        return "WARNING", "Monitor loading and prepare to redistribute demand."
    else:
        return "NORMAL", "No immediate action required."


transformer_data[["severity", "recommendation"]] = transformer_data.apply(
    lambda row: pd.Series(diagnose_transformer(row)),
    axis=1
)

print("Transformer Diagnostic Report")

for _, transformer in transformer_data.iterrows():
    print(f"\n{transformer['transformer_id']} - {transformer['name']}")
    print(f"Loading: {transformer['utilization_percent']:.2f}%")
    print(f"Severity: {transformer['severity']}")
    print(f"Recommendation: {transformer['recommendation']}")

print("\nDiagnostic Summary")
print(transformer_data["severity"].value_counts().to_string())

fig = go.Figure()

fig.add_trace(go.Bar(
    x=transformer_data["transformer_id"],
    y=transformer_data["utilization_percent"],
    text=transformer_data["utilization_percent"].round(1).astype(str) + "%",
    textposition="outside",
    marker_color=[
        "red" if value > 100 else
        "orange" if value >= 90 else
        "steelblue"
        for value in transformer_data["utilization_percent"]
    ],
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Loading: %{y:.2f}%<extra></extra>"
    )
))

fig.add_hline(
    y=100,
    line_dash="dash",
    line_color="red",
    annotation_text="100% Capacity"
)

fig.update_layout(
    title="Smart Grid Transformer Loading Analysis",
    xaxis_title="Transformer",
    yaxis_title="Capacity Utilization (%)",
    yaxis_range=[0, 140],
    template="plotly_dark",
    height=500
)

fig.show()

load_transfers = [
    {
        "source": "TRF-003",
        "destination": "TRF-001",
        "load_mw": 40.0
    },
    {
        "source": "TRF-004",
        "destination": "TRF-002",
        "load_mw": 40.0
    }
]

for transfer in load_transfers:
    source_id = transfer["source"]
    destination_id = transfer["destination"]
    transfer_mw = transfer["load_mw"]

    source_index = transformer_data.index[
        transformer_data["transformer_id"] == source_id
    ][0]

    destination_index = transformer_data.index[
        transformer_data["transformer_id"] == destination_id
    ][0]

    transformer_data.loc[source_index, "actual_load_mw"] -= transfer_mw
    transformer_data.loc[destination_index, "actual_load_mw"] += transfer_mw

transformer_data["utilization_percent"] = (
    transformer_data["actual_load_mw"] /
    transformer_data["capacity_mva"]
) * 100

transformer_data["status"] = transformer_data["utilization_percent"].apply(
    lambda value: "OVERLOADED" if value > 100
    else "WARNING" if value >= 90
    else "NORMAL"
)

print("Transformer Load Redistribution Results")
print(
    transformer_data[
        ["transformer_id", "actual_load_mw",
         "capacity_mva", "utilization_percent", "status"]
    ].round(2).to_string(index=False)
)

print(
    f"\nTotal Transferred Load: "
    f"{sum(item['load_mw'] for item in load_transfers):.2f} MW"
)

print(
    f"Transformers Still Overloaded: "
    f"{(transformer_data['status'] == 'OVERLOADED').sum()}"
)

original_total_load = consumer_data["actual_consumption_mw"].sum()
redistributed_total_load = transformer_data["actual_load_mw"].sum()

overloaded_count = (
    transformer_data["utilization_percent"] > 100
).sum()

negative_load_count = (
    transformer_data["actual_load_mw"] < 0
).sum()

load_difference = abs(original_total_load - redistributed_total_load)

distribution_stable = (
    overloaded_count == 0
    and negative_load_count == 0
    and load_difference < 0.01
)

print("Distribution Network Stability Assessment")
print(f"Original Consumer Demand: {original_total_load:.2f} MW")
print(f"Redistributed Transformer Load: {redistributed_total_load:.2f} MW")
print(f"Load Difference: {load_difference:.4f} MW")
print(f"Overloaded Transformers: {overloaded_count}")
print(f"Transformers With Negative Load: {negative_load_count}")
print(f"Distribution Network Stable: {distribution_stable}")

substation_data = pd.DataFrame(grid_components["substations"])

substation_data["voltage_kv"] = (
    substation_data["voltage_kv"] *
    np.random.normal(1.0, 0.015, len(substation_data))
)

substation_data["frequency_hz"] = np.random.normal(
    60.0, 0.08, len(substation_data)
)

substation_data["voltage_deviation_percent"] = (
    (substation_data["voltage_kv"] -
     pd.Series(
         [component["voltage_kv"] for component in grid_components["substations"]]
     )) /
    pd.Series(
        [component["voltage_kv"] for component in grid_components["substations"]]
    )
) * 100

print("Substation Electrical Measurements")
print(
    substation_data[
        ["id", "name", "voltage_kv",
         "voltage_deviation_percent", "frequency_hz"]
    ].round(3).to_string(index=False)
)

def assess_electrical_conditions(row):
    voltage_deviation = abs(row["voltage_deviation_percent"])
    frequency_deviation = abs(
        row["frequency_hz"] - SIMULATION_CONFIG["nominal_frequency_hz"]
    )

    if voltage_deviation > SIMULATION_CONFIG["voltage_tolerance_percent"]:
        voltage_status = "WARNING"
    else:
        voltage_status = "NORMAL"

    if frequency_deviation > SIMULATION_CONFIG["frequency_tolerance_hz"]:
        frequency_status = "WARNING"
    else:
        frequency_status = "NORMAL"

    overall_status = (
        "WARNING"
        if voltage_status == "WARNING" or frequency_status == "WARNING"
        else "NORMAL"
    )

    return pd.Series({
        "voltage_status": voltage_status,
        "frequency_status": frequency_status,
        "overall_status": overall_status
    })


substation_data[
    ["voltage_status", "frequency_status", "overall_status"]
] = substation_data.apply(
    assess_electrical_conditions,
    axis=1
)

print("Electrical Grid Diagnostic Report")
print(
    substation_data[
        ["id", "name", "voltage_status",
         "frequency_status", "overall_status"]
    ].to_string(index=False)
)

print("\nDiagnostic Summary")
print(substation_data["overall_status"].value_counts().to_string())

fig = go.Figure()

fig.add_trace(go.Bar(
    x=substation_data["name"],
    y=substation_data["voltage_deviation_percent"],
    name="Voltage Deviation",
    marker_color="steelblue",
    text=substation_data["voltage_deviation_percent"].round(2).astype(str) + "%",
    textposition="outside"
))

fig.update_layout(
    title="Substation Voltage Deviation Monitoring",
    xaxis_title="Substation",
    yaxis_title="Voltage Deviation (%)",
    yaxis=dict(range=[-5, 5]),
    template="plotly_dark",
    height=450
)

fig.add_hline(
    y=5,
    line_dash="dash",
    line_color="red",
    annotation_text="Upper Voltage Limit"
)

fig.add_hline(
    y=-5,
    line_dash="dash",
    line_color="red",
    annotation_text="Lower Voltage Limit"
)

fig.show()

fig_frequency = px.line(
    substation_data,
    x="name",
    y="frequency_hz",
    markers=True,
    title="Substation Frequency Telemetry",
    template="plotly_dark"
)

fig_frequency.add_hline(
    y=60,
    line_dash="dash",
    line_color="green",
    annotation_text="Nominal Frequency"
)

fig_frequency.update_layout(
    xaxis_title="Substation",
    yaxis_title="Frequency (Hz)",
    height=450
)

fig_frequency.show()

fault_scenarios = [
    {
        "fault_id": "FLT-001",
        "component_id": "TRF-003",
        "fault_type": "TRANSFORMER_OVERLOAD",
        "severity": "HIGH",
        "description": "Transformer operating above rated capacity."
    },
    {
        "fault_id": "FLT-002",
        "component_id": "SUB-002",
        "fault_type": "VOLTAGE_DEVIATION",
        "severity": "MEDIUM",
        "description": "Substation voltage outside normal operating range."
    },
    {
        "fault_id": "FLT-003",
        "component_id": "GEN-003",
        "fault_type": "GENERATOR_FAILURE",
        "severity": "CRITICAL",
        "description": "Wind generation unit unexpectedly offline."
    }
]

fault_data = pd.DataFrame(fault_scenarios)

fault_data["status"] = "ACTIVE"
fault_data["detection_time"] = datetime.now()

print("Grid Fault Management System")
print(f"Registered Fault Scenarios: {len(fault_data)}")

print(
    fault_data[
        ["fault_id", "component_id", "fault_type",
         "severity", "status"]
    ].to_string(index=False)
)

severity_priority = {
    "CRITICAL": 1,
    "HIGH": 2,
    "MEDIUM": 3,
    "LOW": 4
}

fault_data["priority"] = fault_data["severity"].map(severity_priority)

fault_data = fault_data.sort_values(
    by="priority"
).reset_index(drop=True)

def generate_fault_response(row):
    responses = {
        "GENERATOR_FAILURE": "Dispatch replacement generation and investigate equipment failure.",
        "TRANSFORMER_OVERLOAD": "Redistribute connected load and inspect transformer conditions.",
        "VOLTAGE_DEVIATION": "Investigate voltage regulation and substation equipment."
    }

    return responses.get(
        row["fault_type"],
        "Investigate fault and notify grid operations."
    )

fault_data["recommended_action"] = fault_data.apply(
    generate_fault_response,
    axis=1
)

print("Automated Fault Prioritization Report")

for _, fault in fault_data.iterrows():
    print(f"\nFault: {fault['fault_id']}")
    print(f"Component: {fault['component_id']}")
    print(f"Severity: {fault['severity']}")
    print(f"Priority: {fault['priority']}")
    print(f"Recommended Action: {fault['recommended_action']}")

def process_fault(row):
    if row["severity"] == "CRITICAL":
        return "EMERGENCY RESPONSE INITIATED"
    elif row["severity"] == "HIGH":
        return "CORRECTIVE ACTION INITIATED"
    elif row["severity"] == "MEDIUM":
        return "INVESTIGATION INITIATED"
    else:
        return "MONITORING"


fault_data["response_status"] = fault_data.apply(
    process_fault,
    axis=1
)

fault_data["status"] = "UNDER INVESTIGATION"

print("Grid Fault Response Report")

print(
    fault_data[
        ["fault_id", "component_id", "severity",
         "response_status", "status"]
    ].to_string(index=False)
)

print("\nFault Response Summary")
print(f"Total Faults: {len(fault_data)}")
print(
    f"Emergency Responses: "
    f"{(fault_data['response_status'] == 'EMERGENCY RESPONSE INITIATED').sum()}"
)
print(
    f"Corrective Actions: "
    f"{(fault_data['response_status'] == 'CORRECTIVE ACTION INITIATED').sum()}"
)
print(
    f"Investigations: "
    f"{(fault_data['response_status'] == 'INVESTIGATION INITIATED').sum()}"
)

connection = sqlite3.connect("smart_grid_telemetry.db")

connection.execute("""
CREATE TABLE IF NOT EXISTS grid_telemetry (
    record_id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    total_generation_mw REAL,
    total_demand_mw REAL,
    battery_output_mw REAL,
    battery_charge_percent REAL,
    power_balance_mw REAL,
    grid_status TEXT
)
""")

connection.execute("""
CREATE TABLE IF NOT EXISTS fault_records (
    fault_id TEXT PRIMARY KEY,
    component_id TEXT,
    fault_type TEXT,
    severity TEXT,
    priority INTEGER,
    response_status TEXT,
    record_timestamp TEXT
)
""")

connection.commit()

print("Smart Grid Telemetry Database Initialized")
print("Database: smart_grid_telemetry.db")
print("Tables created: grid_telemetry, fault_records")

telemetry_record = {
    "timestamp": datetime.now().isoformat(),
    "total_generation_mw": grid_metrics["total_generation_mw"],
    "total_demand_mw": grid_metrics["total_demand_mw"],
    "battery_output_mw": battery_output,
    "battery_charge_percent": storage_system["state_of_charge_percent"],
    "power_balance_mw": grid_metrics["power_balance_mw"],
    "grid_status": grid_metrics["grid_status"]
}

connection.execute("""
INSERT INTO grid_telemetry (
    timestamp,
    total_generation_mw,
    total_demand_mw,
    battery_output_mw,
    battery_charge_percent,
    power_balance_mw,
    grid_status
)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", tuple(telemetry_record.values()))

connection.commit()

print("Grid telemetry successfully recorded.")
print(pd.DataFrame([telemetry_record]).to_string(index=False))

for _, fault in fault_data.iterrows():
    connection.execute("""
    INSERT OR REPLACE INTO fault_records (
        fault_id,
        component_id,
        fault_type,
        severity,
        priority,
        response_status,
        record_timestamp
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        fault["fault_id"],
        fault["component_id"],
        fault["fault_type"],
        fault["severity"],
        fault["priority"],
        fault["response_status"],
        datetime.now().isoformat()
    ))

connection.commit()

print("Fault records successfully stored.")
print(f"Records saved: {len(fault_data)}")

saved_faults = pd.read_sql_query(
    "SELECT * FROM fault_records ORDER BY priority",
    connection
)

display(saved_faults)

telemetry_history = pd.read_sql_query("""
SELECT *
FROM grid_telemetry
ORDER BY record_id DESC
""", connection)

fault_history = pd.read_sql_query("""
SELECT *
FROM fault_records
ORDER BY priority
""", connection)

print("SMART GRID HISTORICAL TELEMETRY REPORT")
print(f"Telemetry records: {len(telemetry_history)}")
print(f"Fault records: {len(fault_history)}")

display(telemetry_history)

print("\nGRID OPERATING SUMMARY")
print(f"Average generation: {telemetry_history['total_generation_mw'].mean():.2f} MW")
print(f"Average demand: {telemetry_history['total_demand_mw'].mean():.2f} MW")
print(f"Average battery output: {telemetry_history['battery_output_mw'].mean():.2f} MW")
print(f"Latest grid status: {telemetry_history.iloc[0]['grid_status']}")

historical_records = []
simulated_soc = 80.0

for hour_offset in range(24):
    simulated_hour = hour_offset % 24

    hourly_demand = sum(
        simulate_demand(
            simulated_hour,
            row["demand_mw"],
            row["category"]
        )
        for _, row in consumer_data.iterrows()
    )

    hourly_generation = sum(
        simulate_generation(simulated_hour).values()
    )

    shortage = max(0, hourly_demand - hourly_generation)

    available_energy = storage_system["capacity_mwh"] * simulated_soc / 100
    battery_output_mw = min(
        shortage,
        storage_system["maximum_discharge_mw"],
        available_energy
    )

    simulated_soc -= (
        battery_output_mw / storage_system["capacity_mwh"]
    ) * 100

    power_balance = hourly_generation + battery_output_mw - hourly_demand

    if abs(power_balance) <= 1:
        status = "BALANCED"
    elif power_balance < 0:
        status = "DEFICIT"
    else:
        status = "SURPLUS"

    record = (
        (datetime.now() - timedelta(hours=23-hour_offset)).isoformat(),
        hourly_generation,
        hourly_demand,
        battery_output_mw,
        simulated_soc,
        power_balance,
        status
    )

    historical_records.append(record)

connection.executemany("""
INSERT INTO grid_telemetry (
    timestamp,
    total_generation_mw,
    total_demand_mw,
    battery_output_mw,
    battery_charge_percent,
    power_balance_mw,
    grid_status
)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", historical_records)

connection.commit()

print("24-hour historical simulation completed.")
print(f"New telemetry records: {len(historical_records)}")
print(f"Final simulated battery charge: {simulated_soc:.2f}%")

historical_data = pd.read_sql_query("""
SELECT * FROM grid_telemetry
ORDER BY record_id DESC
LIMIT 24
""", connection)

display(historical_data.head(10))

anomaly_data = pd.read_sql_query("""
SELECT
    total_generation_mw,
    total_demand_mw,
    battery_output_mw,
    battery_charge_percent,
    power_balance_mw
FROM grid_telemetry
ORDER BY record_id DESC
LIMIT 24
""", connection)

anomaly_features = anomaly_data[
    [
        "total_generation_mw",
        "total_demand_mw",
        "battery_output_mw",
        "battery_charge_percent",
        "power_balance_mw"
    ]
]

anomaly_model = IsolationForest(
    contamination=0.15,
    random_state=42
)

anomaly_data["anomaly_prediction"] = anomaly_model.fit_predict(
    anomaly_features
)

anomaly_data["condition"] = anomaly_data["anomaly_prediction"].map({
    1: "NORMAL",
    -1: "ANOMALY"
})

anomaly_count = (anomaly_data["condition"] == "ANOMALY").sum()

print("AI GRID ANOMALY DETECTION REPORT")
print(f"Telemetry records analyzed: {len(anomaly_data)}")
print(f"Unusual operating conditions detected: {anomaly_count}")

display(
    anomaly_data[
        [
            "total_generation_mw",
            "total_demand_mw",
            "battery_charge_percent",
            "power_balance_mw",
            "condition"
        ]
    ]
)

dashboard_data = pd.read_sql_query("""
SELECT *
FROM grid_telemetry
ORDER BY record_id ASC
LIMIT 24
""", connection)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=dashboard_data["timestamp"],
    y=dashboard_data["total_generation_mw"],
    mode="lines",
    name="Generation",
    line=dict(width=3)
))

fig.add_trace(go.Scatter(
    x=dashboard_data["timestamp"],
    y=dashboard_data["total_demand_mw"],
    mode="lines",
    name="Demand",
    line=dict(width=3)
))

fig.update_layout(
    title="Smart Grid | Generation vs Demand",
    xaxis_title="Simulation Time",
    yaxis_title="Power (MW)",
    template="plotly_dark",
    hovermode="x unified",
    height=450
)

fig.show()

battery_fig = go.Figure()

battery_fig.add_trace(go.Scatter(
    x=dashboard_data["timestamp"],
    y=dashboard_data["battery_charge_percent"],
    mode="lines+markers",
    name="Battery Charge",
    fill="tozeroy"
))

battery_fig.update_layout(
    title="Battery Energy Storage | State of Charge",
    xaxis_title="Simulation Time",
    yaxis_title="Charge (%)",
    yaxis=dict(range=[0, 100]),
    template="plotly_dark",
    height=350
)

battery_fig.show()

balance_fig = go.Figure()

balance_fig.add_trace(go.Bar(
    x=dashboard_data["timestamp"],
    y=dashboard_data["power_balance_mw"],
    name="Power Balance"
))

balance_fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="white"
)

balance_fig.update_layout(
    title="Grid Power Balance",
    xaxis_title="Simulation Time",
    yaxis_title="Surplus / Deficit (MW)",
    template="plotly_dark",
    height=350
)

balance_fig.show()

print("SMART GRID ENGINEERING DASHBOARD")
print(f"Telemetry records: {len(dashboard_data)}")
print(f"AI anomalies detected: {anomaly_count}")
print(f"Average generation: {dashboard_data['total_generation_mw'].mean():.2f} MW")
print(f"Average demand: {dashboard_data['total_demand_mw'].mean():.2f} MW")
print(f"Current battery charge: {dashboard_data.iloc[-1]['battery_charge_percent']:.2f}%")
print(f"Current grid status: {dashboard_data.iloc[-1]['grid_status']}")

positions = nx.spring_layout(grid, seed=42, dim=3)

component_colors = {
    "generators": "orange",
    "substations": "cyan",
    "transformers": "yellow",
    "consumers": "lime"
}

fig_3d = go.Figure()

for source, destination in grid.edges():
    x0, y0, z0 = positions[source]
    x1, y1, z1 = positions[destination]

    fig_3d.add_trace(go.Scatter3d(
        x=[x0, x1],
        y=[y0, y1],
        z=[z0, z1],
        mode="lines",
        line=dict(color="gray", width=4),
        hoverinfo="none",
        showlegend=False
    ))

for category, components in grid_components.items():
    component_ids = [component["id"] for component in components]

    fig_3d.add_trace(go.Scatter3d(
        x=[positions[node][0] for node in component_ids],
        y=[positions[node][1] for node in component_ids],
        z=[positions[node][2] for node in component_ids],
        mode="markers+text",
        name=category.capitalize(),
        text=component_ids,
        textposition="top center",
        marker=dict(
            size=9,
            color=component_colors[category],
            line=dict(width=1, color="white")
        ),
        hovertext=[
            grid.nodes[node]["name"]
            for node in component_ids
        ],
        hoverinfo="text"
    ))

fig_3d.update_layout(
    title="Smart Grid Digital Twin | 3D Infrastructure Model",
    template="plotly_dark",
    scene=dict(
        xaxis_title="Grid X Position",
        yaxis_title="Grid Y Position",
        zaxis_title="Grid Z Position",
        bgcolor="#111827"
    ),
    height=750,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig_3d.show()

print("3D SMART GRID DIGITAL TWIN")
print(f"Infrastructure components: {grid.number_of_nodes()}")
print(f"Network connections: {grid.number_of_edges()}")
print("Interactive 3D visualization generated.")

validation_results = {
    "Grid topology": grid.number_of_nodes() == 21 and grid.number_of_edges() == 20,
    "Grid connectivity": nx.is_connected(grid),
    "Generation model": len(generator_data) == 3,
    "Consumer model": len(consumer_data) == 8,
    "Transformer monitoring": len(transformer_data) == 6,
    "Fault management": len(fault_data) == 3,
    "Telemetry database": connection.execute(
        "SELECT COUNT(*) FROM grid_telemetry"
    ).fetchone()[0] > 0,
    "Fault database": connection.execute(
        "SELECT COUNT(*) FROM fault_records"
    ).fetchone()[0] == 3,
    "Machine learning": "anomaly_model" in globals(),
    "3D visualization": "fig_3d" in globals(),
    "Engineering dashboard": "fig" in globals()
}

print("=" * 48)
print("SMART GRID DIGITAL TWIN")
print("FINAL SYSTEM VALIDATION")
print("=" * 48)

for component, passed in validation_results.items():
    status = "PASS" if passed else "FAIL"
    print(f"{component:<28} {status}")

passed_count = sum(validation_results.values())
total_checks = len(validation_results)

print("=" * 48)
print(f"Validation results: {passed_count}/{total_checks} checks passed")
print(f"Project version: {PROJECT_VERSION}")
print(f"Simulation mode: {SIMULATION_MODE}")
print("=" * 48)

Smart Grid Digital Twin environment initialized.
Core simulation and visualization libraries are ready.
Project: Smart Grid Digital Twin
Version: 1.0.0
Simulation Mode: FULLY_SIMULATED
Nominal Frequency: 60.0 Hz
Nominal Voltage: 230.0 kV
Environment initialized successfully.
Grid infrastructure defined successfully.
Generators: 3
Substations: 4
Transformers: 6
Consumers: 8
Total components: 21
Electrical network constructed successfully.
Network nodes: 21
Network connections: 20
Network connected: True
Network density: 0.095


Power generation models initialized.


,id,name,type,capacity_mw,output_mw,utilization_percent,available_capacity_mw,status
0,GEN-001,North Power Station,thermal,450,350,77.78,100,ONLINE
1,GEN-002,South Solar Farm,solar,250,180,72.00,70,ONLINE
2,GEN-003,West Wind Farm,wind,300,220,73.33,80,ONLINE


Total generation capacity: 1000 MW
Current power generation: 750 MW
Electricity demand models initialized.


,id,name,demand_mw,category,priority,actual_consumption_mw,status
0,LOAD-001,North Residential District,75,Residential,Normal,75,ONLINE
1,LOAD-002,Central Business District,120,Commercial,Normal,120,ONLINE
2,LOAD-003,Industrial Manufacturing Plant,180,Industrial,High,180,ONLINE
3,LOAD-004,South Residential District,65,Residential,Normal,65,ONLINE
4,LOAD-005,West Commercial District,90,Commercial,Normal,90,ONLINE
5,LOAD-006,Regional Hospital,35,Critical Infrastructure,Critical,35,ONLINE
6,LOAD-007,Data Center,110,Critical Infrastructure,Critical,110,ONLINE
7,LOAD-008,Water Treatment Facility,40,Critical Infrastructure,High,40,ONLINE


Total electricity demand: 715 MW
Critical infrastructure demand: 145 MW
Generation surplus: 35 MW
Grid operational assessment
Generation: 750.00 MW
Demand: 715.00 MW
Power balance: 35.00 MW
Reserve margin: 39.86%
Generation utilization: 75.00%
Grid status: STABLE
Demand simulation completed for hour 18:00


... (10 outputs left out)

AI GRID ANOMALY DETECTION REPORT
Telemetry records analyzed: 24
Unusual operating conditions detected: 4


,total_generation_mw,total_demand_mw,battery_charge_percent,power_balance_mw,condition
0,541.788713,643.270394,0.0,-101.481682,NORMAL
1,587.621340,654.209463,0.0,-66.588123,NORMAL
2,588.842504,655.785244,0.0,-66.942740,NORMAL
3,605.104847,667.489398,0.0,-62.384552,NORMAL
4,592.602300,670.183454,0.0,-77.581153,NORMAL
5,581.020690,681.287141,0.0,-100.266451,NORMAL
6,594.867260,683.019098,0.0,-88.151838,NORMAL
7,643.602418,687.625896,0.0,-44.023478,NORMAL
8,724.043039,690.216513,0.0,33.826527,NORMAL
9,780.779817,705.232304,0.0,75.547513,ANOMALY


SMART GRID ENGINEERING DASHBOARD
Telemetry records: 24
AI anomalies detected: 4
Average generation: 637.89 MW
Average demand: 668.20 MW
Current battery charge: 0.00%
Current grid status: DEFICIT


3D SMART GRID DIGITAL TWIN
Infrastructure components: 21
Network connections: 20
Interactive 3D visualization generated.
SMART GRID DIGITAL TWIN
FINAL SYSTEM VALIDATION
Grid topology                PASS
Grid connectivity            PASS
Generation model             PASS
Consumer model               PASS
Transformer monitoring       PASS
Fault management             PASS
Telemetry database           PASS
Fault database               PASS
Machine learning             PASS
3D visualization             PASS
Engineering dashboard        PASS
Validation results: 11/11 checks passed
Project version: 1.0.0
Simulation mode: FULLY_SIMULATED
